# Пример - Комбинаторные потенциалы

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **990f446**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/990f4462b17a1afaad25666220ffc1fbc2cd4d26.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Вычислим комбинаторные потенциалы для заданных типов и сверим их с перебором. Для лесных примеров используем также формы разбиений, стягивание и точные остовные коэффициенты. Расчёты проверяют выбранные примеры. Общие доказательства даны в заметках серии.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform-algebra 990f446](https://github.com/magin67/polyform_algebra/commit/990f4462b17a1afaad25666220ffc1fbc2cd4d26).


## Разбиения и потенциал второго рода

Перечислим три разбиения множества $\{a,b,c,d\}$ на две пары и сравним их число с $s_2([2,2])$. На множествах до пяти элементов сверим формулу по типам. Сумма по всем типам даёт число Белла, а сумма по типам с заданным числом блоков — число Стирлинга второго рода.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB%D1%8B/%D1%80%D0%B0%D0%B7%D0%B1%D0%B8%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D1%8B%D0%B9-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB-%D0%B2%D1%82%D0%BE%D1%80%D0%BE%D0%B3%D0%BE-%D1%80%D0%BE%D0%B4%D0%B0#%D0%BE%D0%B4%D0%B8%D0%BD-%D1%82%D0%B8%D0%BF-%D0%B8-%D1%82%D1%80%D0%B8-%D1%80%D0%B0%D0%B7%D0%B1%D0%B8%D0%B5%D0%BD%D0%B8%D1%8F).

In [3]:
from collections import Counter
from itertools import combinations, permutations
from math import comb, factorial, perm, prod

from sympy.functions.combinatorial.numbers import bell, stirling
from sympy.utilities.iterables import multiset_partitions

from polyform_algebra.combinatorics import (
    first_kind_potential, path_cycle_potential, second_kind_potential,
)


def block_sizes(blocks):
    return tuple(sorted((len(block) for block in blocks), reverse=True))


four = tuple("abcd")
pairs = [blocks for blocks in multiset_partitions(four, 2)
         if block_sizes(blocks) == (2, 2)]
assert len(pairs) == second_kind_potential((2, 2)) == 3
print("s₂([2,2]) =", len(pairs), "|", [" | ".join(map("".join, p)) for p in pairs])

for n in range(1, 6):
    labels = tuple("abcde"[:n])
    counts = Counter(block_sizes(blocks) for blocks in multiset_partitions(labels))
    assert all(count == second_kind_potential(sizes) for sizes, count in counts.items())
    assert sum(counts.values()) == bell(n)
    for m in range(1, n + 1):
        assert sum(count for sizes, count in counts.items() if len(sizes) == m) == stirling(n, m)
print("B₅ =", bell(5), "| S(5,2) =", stirling(5, 2))

s₂([2,2]) = 3 | ['ab | cd', 'ac | bd', 'ad | bc']
B₅ = 52 | S(5,2) = 15


## От разбиений к лесам на пяти вершинах

В типе $[2,1]$ числа обозначают грейды. Для подсчёта разбиений самих вершин используем размеры блоков $[3,2]$. Потенциал $s_2([2,1])=3$ считает разбиения вспомогательного трёхэлементного множества, а на пяти вершинах получаем десять разбиений и тридцать лесов. Слева суммируем произведения рёберных форм всех этих лесов. Справа суммируем формы разбиений и умножаем на $3$. Сравниваем точные коэффициенты.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB%D1%8B/%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB%D1%8B-%D0%B2-%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D1%85-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%85#%D0%BF%D0%BE%D1%87%D0%B5%D0%BC%D1%83-%D1%80%D0%B0%D0%B7%D0%B1%D0%B8%D0%B5%D0%BD%D0%B8%D0%B9-%D0%B4%D0%B5%D1%81%D1%8F%D1%82%D1%8C-%D0%B0-%D0%BB%D0%B5%D1%81%D0%BE%D0%B2-%D1%82%D1%80%D0%B8%D0%B4%D1%86%D0%B0%D1%82%D1%8C).

In [4]:
from functools import reduce

from polyform_algebra.core import (
    Context, Network, contract, equivalent, form_product, network,
    partition_form, spanning_tree_weight,
)


def partition(context, blocks=()):
    return partition_form(context, blocks).value


def complete_graph(labels):
    return network(labels, tuple((a, b, 1) for a, b in combinations(labels, 2)))


def product_of_edges(context, edges):
    return reduce(
        lambda form, edge: form_product(form, partition(context, (edge,))).value,
        edges, partition(context),
    )


five = tuple("abcde")
context = Context(five)
zero = 0 * partition(context)
type_partitions = [
    tuple(map(tuple, blocks)) for blocks in multiset_partitions(five, 2)
    if block_sizes(blocks) == (3, 2)
]
E5_21 = sum((partition(context, blocks) for blocks in type_partitions), start=zero)

forest_edges = []
for edges in combinations(combinations(five, 2), 3):
    forest = Network(context, tuple((a, b, 1) for a, b in edges))
    if block_sizes(forest.components) == (3, 2):
        forest_edges.append(edges)
forest_sum = sum((product_of_edges(context, edges) for edges in forest_edges), start=zero)

assert second_kind_potential((2, 1)) == 3
assert len(type_partitions) == second_kind_potential((3, 2)) == 10
assert len(forest_edges) == 3 * len(type_partitions) == 30
assert equivalent(forest_sum, 3 * E5_21).value
print("s₂([2,1]) = 3 | #E₅[2,1] = 10 | 30 forest terms = 3 E₅[2,1]")

s₂([2,1]) = 3 | #E₅[2,1] = 10 | 30 forest terms = 3 E₅[2,1]


## Соединение заданных компонент

Сохраняем заданные рёбра $ab$ или $ab,cd$ и перечисляем пары новых рёбер, которые превращают исходный лес в дерево. По отдельности считаем случаи с общей вершиной новых рёбер и без неё. Итог сверяем с формулой Кэли–Муна и остовным коэффициентом полного графа после стягивания заданных блоков.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB%D1%8B/%D1%82%D0%B5%D0%BE%D1%80%D0%B5%D0%BC%D0%B0-%D0%BA%D1%8D%D0%BB%D0%B8-%D0%B8-%D1%81%D0%BE%D0%B5%D0%B4%D0%B8%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85-%D0%BA%D0%BE%D0%BC%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82#%D1%81%D1%82%D1%8F%D0%B3%D0%B8%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5-%D0%B2%D0%BE-%D0%B2%D0%B7%D0%B2%D0%B5%D1%88%D0%B5%D0%BD%D0%BD%D1%8B%D0%B9-%D0%B3%D1%80%D0%B0%D1%84).

In [5]:
def completions(labels, groups):
    complete = complete_graph(labels)
    contracted = contract(complete, groups).value.network
    sizes = tuple(map(len, groups.values()))
    expected = prod(sizes) * len(labels) ** (len(sizes) - 2)
    assert spanning_tree_weight(contracted).value == expected

    # In both examples each original component is a singleton or a fixed edge.
    prescribed = tuple(tuple(block) for block in groups.values() if len(block) == 2)
    candidates = [edge for edge in combinations(labels, 2) if edge not in prescribed]
    distribution = Counter()
    for added in combinations(candidates, len(sizes) - 1):
        tree = Network(complete.context, tuple((a, b, 1) for a, b in prescribed + added))
        if len(tree.components) == 1:
            new_edges = Network(complete.context, tuple((a, b, 1) for a, b in added))
            positive_grades = tuple(size - 1 for size in block_sizes(new_edges.components) if size > 1)
            distribution[positive_grades] += 1
    assert sum(distribution.values()) == expected
    return expected, (distribution[(2,)], distribution[(1, 1)])


assert completions(four, {"A": ("a", "b"), "B": ("c",), "C": ("d",)}) == (8, (6, 2))
assert completions(five, {"A": ("a", "b"), "B": ("c", "d"), "C": ("e",)}) == (20, (12, 8))
print("T([2,1,1]) = 6 + 2 = 8 | T([2,2,1]) = 12 + 8 = 20")

T([2,1,1]) = 6 + 2 = 8 | T([2,2,1]) = 12 + 8 = 20


## Лесные коэффициенты и числа Стирлинга

Для каждого $1\leqslant k\leqslant6$ считаем $\operatorname{St}(k,m)$ тремя способами. Сначала суммируем по размерам обычных блоков с множителем Кэли для каждого добавленного корня. Затем строим строку $r$-чисел Стирлинга, начиная с $r=k+1$ отдельных маркеров, и отдельно вычисляем конечную разность многочлена Абеля. Выводим строки для $k=3$ и $k=4$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB%D1%8B/%D1%83%D0%B1%D1%8B%D0%B2%D0%B0%D1%8E%D1%89%D0%B8%D0%B5-%D1%84%D0%B0%D0%BA%D1%82%D0%BE%D1%80%D0%B8%D0%B0%D0%BB%D1%8B,-%D0%BA%D0%BE%D1%80%D0%BD%D0%B5%D0%B2%D1%8B%D0%B5-%D0%BB%D0%B5%D1%81%D0%B0-%D0%B8-%D1%87%D0%B8%D1%81%D0%BB%D0%B0-%D1%81%D1%82%D0%B8%D1%80%D0%BB%D0%B8%D0%BD%D0%B3%D0%B0#%D0%BF%D0%BE%D1%87%D0%B5%D0%BC%D1%83-%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82%D1%8B-%D1%8F%D0%B2%D0%BB%D1%8F%D1%8E%D1%82%D1%81%D1%8F-r-%D1%87%D0%B8%D1%81%D0%BB%D0%B0%D0%BC%D0%B8-%D1%81%D1%82%D0%B8%D1%80%D0%BB%D0%B8%D0%BD%D0%B3%D0%B0).

In [6]:
def restricted_stirling(markers, positions):
    # Start with the markers in separate blocks. Each new position joins
    # one of the existing blocks or forms a new block of its own.
    row = {markers: 1}
    for _ in range(positions):
        next_row = Counter()
        for blocks, count in row.items():
            next_row[blocks] += blocks * count
            next_row[blocks + 1] += count
        row = next_row
    return row


for k in range(1, 7):
    labels = tuple("abcdef"[:k])
    types = {block_sizes(blocks) for blocks in multiset_partitions(labels)}
    row = restricted_stirling(markers=k + 1, positions=k - 1)
    coefficients = []
    for m in range(1, k + 1):
        by_type = sum(second_kind_potential(sizes) * prod((size + 1) ** (size - 1) for size in sizes)
                      for sizes in types if len(sizes) == m)
        difference = sum((-1) ** (m-i) * comb(m, i) * i * (i + k) ** (k - 1)
                         for i in range(m + 1))
        assert by_type == row[k + m]
        assert factorial(m) * by_type == difference
        coefficients.append(by_type)
    if k in (3, 4):
        print(f"St({k}, m) =", tuple(coefficients))
        assert tuple(coefficients) == {3: (16, 9, 1), 4: (125, 91, 18, 1)}[k]

St(3, m) = (16, 9, 1)
St(4, m) = (125, 91, 18, 1)


## Циклы и потенциал первого рода

Перебираем перестановки четырёх букв и группируем их по длинам циклов. Сверяем результат с $s_1$ и с произведением $s_2$ на число циклических порядков внутри блоков. На пяти буквах отдельно считаем перестановки без неподвижных точек.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB%D1%8B/%D1%86%D0%B8%D0%BA%D0%BB%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D1%80%D0%B0%D0%B7%D0%B1%D0%B8%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D1%8B%D0%B9-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB-%D0%BF%D0%B5%D1%80%D0%B2%D0%BE%D0%B3%D0%BE-%D1%80%D0%BE%D0%B4%D0%B0#%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB-%D1%86%D0%B8%D0%BA%D0%BB%D0%BE%D0%B2%D0%BE%D0%B3%D0%BE-%D1%82%D0%B8%D0%BF%D0%B0).

In [7]:
def cycle_type(labels, images):
    permutation = dict(zip(labels, images))
    remaining = set(labels)
    lengths = []
    while remaining:
        vertex = min(remaining)
        length = 0
        while vertex in remaining:
            remaining.remove(vertex)
            vertex = permutation[vertex]
            length += 1
        lengths.append(length)
    return tuple(sorted(lengths, reverse=True))


types4 = Counter(cycle_type(four, images) for images in permutations(four))
for sizes, count in sorted(types4.items(), reverse=True):
    assert count == first_kind_potential(sizes)
    assert count == second_kind_potential(sizes) * prod(factorial(size - 1) for size in sizes)
    print(f"s₁{list(sizes)} =", count)
assert sum(types4.values()) == factorial(len(four))

derangements5 = sum(all(a != b for a, b in zip(five, images)) for images in permutations(five))
assert derangements5 == first_kind_potential((5,)) + first_kind_potential((3, 2)) == 44
print("D₅ =", derangements5)

s₁[4] = 6
s₁[3, 1] = 8
s₁[2, 2] = 3
s₁[2, 1, 1] = 6
s₁[1, 1, 1, 1] = 1
D₅ = 44


## Пути, циклы и истории построения

Сначала проверяем семь малых типов состояний. Для цикла длины два и отдельной открытой дуги перечисляем способы распределить три метки дуг по компонентам. Затем на вершинах $a,b,c$ перебираем ориентированные пути из двух дуг и все порядки добавления этих дуг. Проверка различает потенциал $u=2$, число графов $N_3=6$ и число историй $H_3=12$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D1%8B%D0%B5-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB%D1%8B/%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB-%D1%81%D0%BE%D1%81%D1%82%D0%BE%D1%8F%D0%BD%D0%B8%D0%B9-%D0%B8%D0%B7-%D0%BF%D1%83%D1%82%D0%B5%D0%B9-%D0%B8-%D1%86%D0%B8%D0%BA%D0%BB%D0%BE%D0%B2#%D0%B3%D1%80%D0%B0%D1%84%D1%8B-%D0%BD%D0%B0-%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B0%D1%85-%D0%B8-%D0%B8%D1%81%D1%82%D0%BE%D1%80%D0%B8%D0%B8-%D0%BF%D0%BE%D1%81%D1%82%D1%80%D0%BE%D0%B5%D0%BD%D0%B8%D1%8F).

In [8]:
examples = (
    ((1, 1), (), 1), ((1,), (1,), 2), ((2,), (), 1), ((), (2,), 2),
    ((), (1, 1), 1), ((2,), (1,), 3), ((1,), (2,), 6),
)
assert all(path_cycle_potential(cycles, paths) == count for cycles, paths, count in examples)
arc_labels = tuple("ijk")
mixed_arc_types = {
    (tuple(sorted(cycle)), path) for *cycle, path in permutations(arc_labels)
}
assert len(mixed_arc_types) == path_cycle_potential((2,), (1,)) == 3

vertices = tuple("abc")
directed_paths = {((a, b), (b, c)) for a, b, c in permutations(vertices)}
histories = {history for edges in directed_paths for history in permutations(edges)}
u = path_cycle_potential(path_lengths=(2,))
assert len(histories) == u * perm(len(vertices), 3) == 12
assert len(directed_paths) == len(histories) // factorial(2) == 6
print("u([2, open 1]) = 3 | two-arc path: u = 2, N₃ = 6, H₃ = 12")

u([2, open 1]) = 3 | two-arc path: u = 2, N₃ = 6, H₃ = 12
